# Drum: konteks maksimal 90 detik lalu stitching

Eksperimen pembanding setelah drum full-song satu konteks ditolak. Pakai **VM Colab aktif yang memiliki runtime/bobot ACE-Step**; salin sel ini ke notebook aktif agar cache tetap dipakai. Helper tersemat; tidak perlu ZIP tools. Piano asli dipakai kembali jika tersedia.

Setting model tetap sama: acoustic drum caption, Base / LEGO / CUDA FP32 / seed 0 / CFG 7 / 50 langkah. Batas **90 detik termasuk margin enam detik dan padding model**, bukan core 90 detik ditambah margin. Untuk piano 4:03: tiga konteks model 90/89/90 detik dan dua sambungan. Crossfade float equal-power .58 detik; kualitas belum terbukti. Groove/timbre antar-chunk masih dapat berubah.

Tidak mengulang bass atau mengganti baseline 30 detik. DRY_RUN=True hanya membuat plan. Notebook ini tidak memasang atau mengunduh ulang model. VM kosong perlu sel setup/runtime/bobot notebook 12 terlebih dahulu, tanpa menjalankan generasi bassnya.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, importlib.util, json, shutil, sys, tempfile, uuid, zipfile
from google.colab import files
DRY_RUN = False
if not DRY_RUN:
    assert shutil.which("nvidia-smi"), "Aktifkan GPU dan gunakan VM dengan runtime/bobot ACE-Step yang sudah siap"
for dependency in ("numpy", "scipy", "soundfile"):
    assert importlib.util.find_spec(dependency), f"Kernel notebook memerlukan {dependency}"
# BEGIN EMBEDDED CONTEXT HELPERS
EMBEDDED_HELPERS = {
  "electric_full_song.py": {
    "sha256": "1141f65172b7916649d87ac6baa82cfd9115a9cf7afe853d3c2b3bd8bbde1b16",
    "gzip_base64": "H4sIAAAAAAAC/70863Lbxnr/9RRbnB8BFQqk7DpnyoSZUWQpVo8taSw77VTVYEBgSW4EAjgASJlWNdN36Bv2SfpddoHFhZTipvVkYmLv+91va8dxblSyiOVRFhSFOB7/93/+1w/jcSHOYhmWuQrFL9A+FCoJ43UEI8W6kCIoRCDiNJFincVpEMlInKZxMBNLGWcy9w4OPheyEOVSCvlFFSXOy1SSwLh8nZRqBUskkZDJQsEaJ9cXQ/GgymW6LoUqCxGpYJGkMC0Up+8vvisOjsej13CmMMg8cQUzinSdh1KEaVLKL+VQ4ElWaSRjGBLHP4okhSGwZ7hcJ/e4eZoLWK4Ml/DhHTiOc6BWWZqXIsgXWZAX0nyHabY1v3+HNczvVVAuze+0OJjn6Upk0BarmdDN19aQYj3L8jSURVG1bKufeH9eodxmACbdfAN/x/IyWMkiC8LqROu1iszvh2BTtX9V2VzF8uDg48mnMzEVr1+Nx+OD649nN2ef4PPxQMAfpwiXchX4G5kXKk2ciTgeCkdF8MORGsNHM8CwA80EQeyB3YtSZkeb4zfYKZ0hLxbJjQoljgjXUYBTslyGSq/szIEUytevsF0lRZmvVzIpscNsUEgZFdBwO77TKy7gckESSr8AxOHKf/XGNH0uc0ntcA6c8gab0zluEfll6ofZGlo/5WupVwLaKPU5LgqgxRJobQ7YhouX2xjo0NAzHkbAvmWQiyAM01UWJApP6okTEeXboUiCcp0HscjidXgPy5RIXg5tY/1xkGJFGMsgj7dAR0Cta942KMsgvAemKZaIqCQtpchh/wB5Auk+AE6axcg8wCxiGeSrNFFh4fVsciIKIguxWhcKYETHj5FrVAI8mKT5Sjce6TvlcgEcB+vO0zhOH5D6kQ8zFSTpd0XPFrw/3BvoEkfjCaNtEqzwSID6p4ODg0jOgSuLMAVC8vM0LV2ki6EA/AOmchlNLwFIgwnjwnHOFV5TzHIVLYBPlzK8R+Y2TI4gmaXpvVjE6SyIC+RPoKQ1EOlGAjSBGpIEz2KYGlaAszh8eNwaKBz5jY4x8HIJKN9Id0D9qyC/B7TDENcBETFXi2IEAJJlQdTsIV8jOZZpGhcjgNXKl5sgHoFo8rItEWqYq6yEPlmuM1+LLOzjDRAaMEFFLooAfWv8k8P4HPASxy51iREQ0woOqAofmdUdAFZyakP86YPyompeQ1OoAmEkEKj16rSivnc1dODJL0DBEYjl3G1DQq+LS/Wel84cKADnR77jWZ6nuTt3Pl5dfRLbADAQqUzFailm6/sgqTHJmJ2IR1zxyRm0QYDN1AaYjIDHS4n4AHjqkyEYgEAjoFMARKTC0kOJeC+3hatpi3AFwhlYc2CdGWcSJGCey0sMzVIjQB2pBsduugEMF7/KROYBaqKjD0hUhRZqnT+HPNFD0nQdVCOrIPNR4PmHI736YNgaBhLT19rI5zH+oTNoQVo1qcYCjRdE3Frjj8GUyyDaAuBu6cb21evJzWvAJob0nI1MojQfnZyeHd2gPD/23ow8aNyMZioZZVtgxsSxiLO5EsqBvUsRMWSpSoBT2ipjRJzrFcFcljIp0rywN7pj0limoE6ILkBOysjl28Il68tVvBHLxNUTBuJncWwxXZeAnZMIhI+cAcazoE20nrgolEAK999dXH4Cbk4WQNpFEAdL+H+5NjBWk660tP98L1BWeL8DBFygERdU3tBcalAhkNhBt4Liw7uYiwMVSuJxLWEJZj7jxUUpqwmIWwBO2LYXsQ70us6NFl88DwSEdHDbtPBI8kynwgHFzNs7NikcWAJDT65wth/g+ksL7JmM1ysRABIy/N9vHxCanvhnAHGCYqSAMSRbK3MQrAAxS2ew716QO5GqlEcxOh4D04HZ6Rvl7s/Xcewjz3oq2yazoYiDeC1AL4OlKRYkAgrlOQ3c8EU1Ckh3L23gg+HmIUHcTgB5wImAZdfgoaVBUC7Y3VqNOEDvvPo8R+vOlsJkRKBZ56UZUDjOpl4grHzmDMjSBmshndjy3KUmbyFLWhDEGuBnKKrWJFyC/gT0NlqLYJU9qAiUx2Ag/gGUIxqOQ/EK/utVCb8B6Ax6z75kAGFQTGhZyBSMzft3X8X16YfjH7qyvz6Hvu9AXx/YGg0dkIFfpQ/uw8JlWQmmB9jhskTogSE1BVsvWqOwTpPpMXwsApVMx95fh6Axv/hVF/gptcUBJ4xVqEp0SxZHWhaD98E2dAFHTxOwLGZbsIZAuz6kqHp/BLGyCMKtiMH0AccDtt8CqQIJlJW1QYY6EAEimfBt7PVc/n2tckmNNHKDACOzg65RX4Jv0Dx9xWtukGxdsKHBYkZL2N2A5oMVBygHkQ+bXS6cDnQbWtqDagg6J8CoYO4qoIQNGxkb1BJ8pKZgJ18Izid+EmOzwhg+zMnET9PGSfEbQG0Pxetg83FHKthEAz6g4DPxhj/biP1pau/xEyKTNA4tjap9PDy+07RVe1MeyAv31oa868znq0wukPecI/QawUtFuXy0wf9LPAp9cqNymEWZ8Ab9NoBzRP4KjcOD09KlbqkQiI3BHC0NZ5OCvJPTRzz8k7Nr0SDXSyDj8fQQD0Ue01E4YY8qXPnF8Q+x1GOZMwZ3Q1ZjU3R5GCpApGuw/adGquiRIHhwfUNewaxw9cijCvikQfXAPRicO29lmKJ3X9HCShVAbeFyQuSPOj8Sj6b3CQgdKOSR93tqillu1JIAzFe07Cr+B+2crXv53/Ir2OnOwQdJkB0MHxbL4NWbH4biIQdK88lp/2N8C1CymEzz7i4e1N0v4EMmnQa7PcctQZtfapVcYcByDdhEsw5XC5xdx69HNG6wT+dWUqN7xZqedPfxGMVCS3I8KyQayCbhhR7rwzKNUYDESB95DRsTowILAsR8VGgYpblaqCSI/WVQoJvEZGEYncGYzFPoQRLxMJBQuJZkIfbymRC7IgZGzSTLmJZcASf/wQcfJVdyp0/hzNFPL6fmojQxRdnhaGfUkknA6qi7mNM1q28CBXZOjG4vYczFq9zqZZ27W6da+W4AfE7kYxtzTbRVyxHe6sV/QvxBi2sTW0Vo9bifEave+M1exN6QUdbBlkjnbCD8SEhGg41wDEcE7Qw7wWFBiONJVipZgwvwAh6oj/a9wINZ2mzvETmMCKtRpEYi3YEdiueqpZv8Eso8Kw2VEX14q/tIoaWGzmlBmBpypNNP76fnARjWPBzDQ2i48zS0CQ2ZemD0sYERpqsVXgJsyz9RsXWV2N0+ME6qY3wP5+jRdneNs/Ig0n5G+YF9Rgf5Ji1HcNJ7tLS93rKr/8iXrtUfL2ETPZDQIak4lEM8Wpsyh89pvkrxEali8K1E4twhevaSJuhf3pvVbUsLI7W+5ByaCju0WtGpPgq7kIbXwBTlvb83ex+DfWAZCOCeUVB/2pp4aMbTdEYLhzV0EKomaG6vybnHo2HUWC6NYYJha6C1Bw5/0MPxlDKyA2H47YEHCRwYrArXfUV+DDs0GCm+vLo8QzpDVGCINwdqQhjViu5hiUJnFqfhvZhMqxN5GIPQFEUEPDG7kXnBPXnw4NLMRpRNw3LnBOffxw4A1jUwPxT/qM+DMFdIEYyKRx21/60RtNeYhq/HWtpPmnjD0CXyOrTT3087TNGZDEoKxIOucaL0IWk0hEBgkf7qn/8Q5HhWGuOcIIsckXdlX4SyMCoBYw2lKMeWR/dyO6LlxUqWQRSUgefcPTHdVOabWxOXvaCJ29ptg91TtYkZmWmP/bmQnaCsbl4foiDgn2COwaL6p6E1xGezoxrYsEKYsJ/6loa71FP1pH1gGDxpftdgJLKpTKBqpYZRhOdETuxspPmzg2t9y874FptiYCGWPoYhYAwzoUOCw2fyh1b67NnBAr8tJU02yt9D6TKJfM1L9T66oe8uNmPU/CQje4/aPDZ3R5cOOkC77SFTlaBRoAnNoKTh/ZhG7f/ILzIE28bnMAQFjGon6JDyKiDdyB7G3KtJpk1NKo2i/5QgIwWlB3I+tZWK+SgxY4s2Dayjozg68Qo2DHzHW86+Cq11j0CyBZh0oYTa6Pz69auRjqNg+idXkZWL4YNiVoITj5zFwCZjOlKUkdsqb6u6ELsn4O1XOcIIOugHp7McMlF5OoaUXDu1SGGs+nufHv0XArS9M+l08mVbBh9vpvWpHQ/c54Sy97DOYHkZrPqdUR3rBGdho/I0oVMYn7TbRfPsodO+UfbJrMyS67w79999/sU/PTl9R5oQvv/17FP9/eH6/S8np387u3yrv06vLs8vfn178dGxzPq08PRmt7g0BkCtzbntoDnQW2cYuXf1Aa7Oz99fXJ5NnWPY59PHk8ub86uPH84+3tg9Wg02YOjuCXbjkQPMwzQitB5gV+YlGgN2FLZvvla8JiGf5uHSthypwcOcD8a/K+/iuSD46ee3J2Kd1N4I5vgoIK4xRzrx9Prz6MP1jZgHcTwLwnuncRadSPHAuItioFgMAAOb6097JM6FqxXm4JqpNd9ZAeOKxHAxHnWgvRPMhZ9z2v4d7+Bau3knoUSQ6S7r9ijDfMrFr0A++iyakcj9OinkFjKegzQL8gWI68PD+wf81c2JcTszt64wIMbmIoNJL6TtmhUtb7TvVAjEgtOMaOBJvAirLYCAGbm6WKExLMvx2M7ttalquBP2Rm/VpxF65qPfTs6mdbHDPF4XS8svaUXDi3VGmdlnwdUGVBPYAB5/E0ifbqFBy9DS8t7s173ebowCNJrYb0ZSNIXBKENrOP8X/m1RkKkV0j1ue4E24fgYhvD9FxFIBT4zaSeYGkdnx4BSzoB6/vIRdI3hFIpkF2L/KfAPd9xWlSYoDJn2rLbOLCDwlrLuT+737dSaSDuSZGu2d6N3mhRa0NgPujag8Hb0pWsuOgQAA1qIp5F/Eacg2NCT0UaOCz7cJlXgVZpiq+/sEi/2HKjC64iNO87GeFoBorjy7MwR7NtJJBlbhCz+hntqGtlcWegqgLQZlKmb28N0pKcvtgM2kcyVVs36nFkcJK7ZUisp24AbYu1Te7rRmJgQ+T2dGRMXUyKGx/3KPJma2ivkahIkU0OEXL+1w2cD3mhUT+mgVbMAa4opNlhl6hQk93yUe07HybKgCMCzLqIt4LpFG1FAbACjVoGbazT9dL+m16Q31cCsJSZbvdVtdwNEcPUPCN9EzQGK1YZ7qoO0bNXQPmgBUUMPKGhaA2NYUeC0JoE+pFartaGPxW8oStY5CHqK/ORTywvkHgQJFTJQIda0NgDLfFvLE02QhgeZ89uIwahSVmKJpzyjn4hSNBJQy1prdUgVnKhyXUydOVg6wFtDnjGdO4+knugLZTYaiL7/BN4wNTUKhVCts+GKMiq2zv6NpGZcKzR3WDJ28sthukYmpITysZ1WelGi6P8jD1nlFAMQrKspOrnsX08f6fBPw1aycX/I1Vr3DycY1bydWgSrjA6xz9O6YnqNwJenMhLkxXwjdTFwEJaVWuLVeUXHoA8kw32wkNq5/sMoQieoiXRyh9yq2tWPsToyocjNsC6C9eGcGyUfvFX2GlHoOiv1hcpJzEj8bgyyTIT/A8Ko2H5vprqB2VjN4GAxwJRWmnHr8T+9um+vuRvnILcAriit+RcV/xRu08PURSUaslXNDSipYh2X3leVOQjBCrbo23cA3sRDA9qDPbe195tLGaF26O5oI6oXdxbyKPahq6m9f1PZOZY7VaDi+zpfKPDIgWTMVFbjL679t2fn74Hx3nK1DljfatMy7OZU3xqDHNuwe36o4YzWL5lwEzORw8b1/mbasPplOKWJoIkdbXJt/V3VuxqlNGqEqyzjZ9SRr41b2CMLhkPfIuPx2D8hnPtY8D0eabIwQ7UaG+m/dTzTVEiRQqkw2xQDjuP8TcpMBKJYgcYQGLlGbUy+NoNDBHPMjVxdfRjhUrB+/bRgFRT0GAArfcJgDbq+imK1kxo7aKxRH4cRB7ILC/c5asKxfwoRmfhOH+7bVaANEraKL3eSGpM6jdbY0FiVOkDZrtY43FWwMTS2v7aXNXH2LIKZpyBzQWjNsFpwoqubrUJm4Ou+iYM/Fp87vfrw4eLTzXBvtcg3BOhYM5EZ62sVYE0B32SxkFFjhypiiEXtEdqGNlXpvIKFLNawSEaXaXkOujIyxTk6fY3T5tgOVhZPN2YW28G0klU8aozexvsP5kKrEN1MtErXMZ1p1U7YgzBr5mNsRNcWcjR4n5nQeF9kYjiRms+xcp8fx4DBUNAhwaUzSZtaEWkdG3G488+u85+06+dNONFsO+jiyL4eFdCbtHHjHZR+E6GfMHEIPs2331ZfoIUe5QzJJMfcFMt52Ms86qGEleXQsS7AIPeEHjx5EUhV/OEy3jrq1zGumo/ZeIXzNEPdkleGWcOWj9VJGj3rc3V35TgZPukqagdY5+2wChjvhc/BsNpTJet0TTG6+lkYJSAjmYERjaxcPwlDajCvwcRvJ2dUfFIqitiC+iB3xBPvgjx6ABSAg1IS8Zs3OLBBQdZLuMVnZaCJEom1AlxmV0pCtOfsSRj1qGdGZY8nZ6X4+svmGmGOoWgVn9rlubhD24XDkEVCE8ksmDbTV996A4tztCqY9IRa5871+5PLCb3fezQb337XTP19d/dUGJxSAP3X68+ieiH2fACWj3XQ3vnq8kxcn9zcTOyd6z0n3uv5ky6+Hj5zOB3jZwqtXx9WNLbjjFbhD1rn3ep/XU5OKtEnYeP7lmYkh+KIU4hk8R/hLKfOfvAAwFnD/h/UQUoiFQqYseCT+dwndwxDrjXdVIqsLsYBPTjdmY7quhU1+d06fGATEKGgZs/upopNa89OwM1SRG5fIK9jyw4aSsomUmskJSG0JOUkhDGBIgcTkfgKpR5+64D2LrDkDkYev7iGPiWRhT49VdRbGQYmJJVs0pCjkfU5YScMM3b2vh3fWU+j6MlTf2SzY5c7FmbQQKe4qrFJqpdV8HtIoeoGyPQMk+MldUxG/o4sDo6AlW5bFQV3OKJirXbncwC1czP6QLwzOzhoM6i5Ykhqx8ICKAe3qZwE9BE5vLgEnEKWJRW33N06PAj1W18vT7WC/tKgGBVy/SKVtaqVJdBvYbGuYNj3CHbXE9j+ahy7RqKCZat8Arcswf+h0D7qzVgu0uZL3SBuvKjtbAMGNfhZPj85VV/NlmSN4Oog6dC9sprAxQKGLv0aEtzTv74ZjXcoelbBRzzrYCFN15NNkPiIgpFFJHgvt0R99P6BzEVoGepPMBgNojyFbwPdwbMPYE5Ytzcel9B+wMN5oQ3XHruV7ayRKd5qMBGTj5Y59DSazoyPxPANnNU7wzIFv1BfNVMdtwcgjHIQzOmKeBtG6SQTQeq5y31OKsK14q240Ih2FhXRW8qLvfedkrjfy++XwWThoRrEH4Qt+kGvHHk2XZKGAc3egsBF8MGnAQ9lCODW+LQNNCimQDFbarEmesSN1RpSXvewkG8M03zUkWjtJzV9k4jXeE4jK+uYcnm6UXslJOVQC9K9K4Q9F2GbBy9CYGQYwiaDlz/xqlUTlv6wThrpbIS5ZpOQ3WatfVOot4vS7jqPkdrVblWcyiqyG+zVFC9fslF+3Vq0URx39yzALkjJ4FO7Bb+QQe/O1uZWVnFgq4g6PVG9Ua60CzqpaOihwK6P3xdJrc/dKLW727HV/sV6ehvx2V2b1W/znrshxmA7m7Vi7M9fqd+FqW20YSWqpl2FjUk9omJfU/FU/70r2mzpYJ29s62cTq+zM2ydyeCen5ADL6JtB0bWbAuOYmPBnaNesrBOs+xftzlo97JEhswI0w4vdekHDH7E5K4Jzdj+rj2Z3rCY03ruCTPLHKS7aXszNOTFtDF9nmS+1XttJaIOdnuVL8yi9tPu/yKDqgPcu+z9tgNkS2w9tRHDarpn/EnemaXgzbxeNf6tkMZox67AvyVVCMgVgLHUF2Ss+Iv4CCSg+NVOlVmseHPEkp8tGmkme38wC/2i6+g3/HAWE5GnYhf0x0zhi3eSL8jWvqYeN5IcgcS4ue9HaQj+vTUT/1kIP9BT3Nq/RwqZXlNOAbYMgNGnu2IE+h/QKG6P7/auXP2zGdXae4dzTME6hwmLWj7/rp2oDltPJnOmvsV470zr9Zw9Gf/NqalztVJl6+mgiZKtgi9qtV7hyx9jwe7YQYeqqLqGkAIKOcU8Llxq/8wqCtMzUR+xIoKbz9foEd40imP0uvQXrmxyrujbYHkkb2DXZDQqyGkMkoedOKFGQFX7XxWRDZ/A7FBUr+H4QZg+kWRrx5AI9htkO23FDPeoUkbViYb28vWhTO6IO5sxS26si/FNNoma9cegFdFzbkiiTiqOvDWWMfq+/yG4LgG7zQmQZRWWI7KMJefB5wpNH31p4uSD/wF4HN39Ik0AAA=="
  },
  "drum_probe.py": {
    "sha256": "e1197fddd057474b7cc0b9258bb0851a15996fd9e2d96281a99bd4fdf65dc50c",
    "gzip_base64": "H4sIAAAAAAAC/7Va3XPbNhJ/11+B5B5AthRlx9NmTj4+uI1z07lem0vSduY0Gg5MghJOFMkjQNlOxv/77eKD30rSds4PsQkuFov9/O0yz58/f1U3R1LV5R2XpKxJ1uT5UpbFjkhR7HK+rJiUpMEHovacVKIoeEpuvr9dvlO8It//8uqG1E2hxJGHz58/X4hjVdaKsHpXsVpy9/wf4LnI6hLOYmqfiztiX7yBR0ck940SefvU3IFcCZfSrXwQVSZyvnDPPOeJqkUSo9SxlppJK/hi8ebt7bvb9yQiHxcEfqhI6ZqypGykEskyhXvL5emSBvRYpjzX77iEO8HiN8s7Jjm8SvlJJBzeJU3KaGAYVTVPhBRlAetZXjJ19QJIRSEV8OSFgmXNHRYl56mk683F1u7dNSJlRcJjmbAcGL8ML3BrxmuuV+F8oP8GFssMeaexKuOkauj6fd1wyyRhlTLH/yDLnCkwiLsXwZPJQYAFkqQ8VqwQKFJIwEpwu5oJMF9ACqaamuVkV5fliZN7ofawKTkERBas5oQVKUnyUgLnvVjumQpIVuZ5ed/5AStKUvE6K+sjXoiAB8DLgFT7mml3QR7pY8GOIpEh+T7nrDbS7YWSAVEMLguGI2DSXGpqWYEJyB1X95wXhhGHreii4JxF/giCl+RUguq0sx55XqZw5071MgQjPS0Wi5Rn4GnJge24VzaqauAGsknQm/y11iIIxiNtqDgDK90BcQgORvVLrRDrbuG/RfUafls+K9wY0HsaoH5BqegJUUv7w5v41e3rH2/e377y0RtZnezFiZsz8QcUBqYAq8EqSE42RgYdgiGGCbgNsIXo4elKFHCiW93xgtcMLb/iD6B6Y1nzsmU/+elvk0baGYYXFxfxTazdNkanvVjBzZr8C9ijMzY1X9nfjrdhBRc5CX4fHqsrDDTxMFjZrgd8RUacjp2C/FDIGBXr+WuryfC+FoqPCYN2x8KystbujviUTY0N7JXRC/6Egb/UyFpD4T07Gc3ov0YawZ8vvbbxeWtS7kFgq0CWTZ3wwAbAV4FUrFbRRZA2xuzR1UXAH3jSKB69hqBCfi4VsSYVZfRTWXAbMFOO0Qazt1f5IWipzE9gJX3zCq88I4G/1YxMgg6ZUizZazJjNFMeakhIhc4obVVgL775NtD3j3UdaYlt4Yl5cRJ1WWA8uF0zr7p9kC440+ZAF+ltifNyt+NpR3lkSXxf1gfIxfd9OnANSHs9Qp1sax5D4ZCQs4tWfJusNRdeg4L/2wigy6qrF73dUF7itq64ra9//Pnm/dWL+Lef3/7j9u3g2h33ZM/T+D/lXVDlzKgmKYtM7CJU7IqaB7kaFj7jgy5UDE0XajoZI0WIFUh69j3oDMoRf1Ce7z+LTHnt/BVKi+TkV5Y3/Lauy9qjFlpwyRVJRQaOJa/Ncw1RgVWkzFPSpTJ9bANMGCn4vd25OsE2LHbGR4wjhcdDKmoPFQ45P9KlkT8IqeLyYBzZUEMaBSVFHyk4vmqgsNrMCg54rmKbU+naXG+Y+DRSAFrjwDFCIwcfYqhJuYzRuggiAGpcjvYWpcK9P4NngD6VKBowiCmI+AxqvcbqplFMsm+KA9ZQCCYwmoIMUOxC8uvNLTjkI5TaHLOK4nUBpz6G5KeSoN/BfbuUjns1sIA6ApUSiFffvfknVHUmwP70ydS5NqiGmbCfqIwSjUJV/dhZHJiylCkW2YC2VcsbxPyqLWbUph/9b5eC3B/+olcHRmmICAm6UQSz0TBDjvOVTkijxV56Guz9C3ldc/6Bd0zI3aPiUtuhQFyTiw9atoDc/isAMoNxULM1+CrGbV1W4TDz1+UHXkT28h5tedOvR2KFsskyyPuQWXgNITVgU9ZiJ8C64FFZvGdyH5k0OLnbYJcB0GFSVo86kkfEgRHOH1ddy9q+fRZNDl9Pg/ttq7NkzwrImQQMicrBw6n/SSOZg0Yk6GQbOiKlWwhe7BjoGmLVSQi4WktM1xNJnxZDjmFTpVgOTQKIKCbJAn1RI6vIuXAn7++NCKtCx2hj04Hzarr9GxRZndlmfHrqz7auRL0aYwqpzUfToHIrXeTTtqbb3/7cEV9HG7pcHtnDEorHUnJIQimms6sL+Ge5dI3K0jQqAX2ply3KW6ZAn/IaY7puFWVfUlvne+qxgkzRjU0dRmnVo9qD3jtEMKeZjf0Deyq6TAClDQqsNyyaVm+btmfa+sFXjsPlejuUtEMAnqUJAD9EM1jCCBlA8U0Ouv50AvP83IXbK6DSzt7c1zrViSzWBTmOexnMRyuYq56D5UCAjIxtjJywBEbqW8v/v9xcjm7cOWU046fX3d+2og/N3kGDCOPWmzo/9uwOStMAXOIcA5cHwN8RLMXW36NvLwAN6ya/q92R7faBd6zNEzknMpOC7SfaIZO8B627gSfDxj96GSCvaAAlhrL3ElGvT6OTzq9bGCQkLJfOCaFwNXVhM+JiRrs9rOdN7TTpNv0BFFwM4ryl3HHlOdwFNcXxA6th/cx54XXEGwo2kRCbz6LLUX2fFJ5b2JXgyKPUSOpY5RyfNJACLj0twlM0PQOc5DoD4MnrGZec9sG0V1CwO4w6xO0ZNgGCb90nDREMUrskryGD6S61XuxcCfWifQ3fAxvzsodLkcA4t6XYUIOuYlsBt8+irvSMXn1OkxqfG0nMoMH2/AQgt8hEYmBkxiAFpT21Quixowxcl6G3wNlcAabdoYKpoYA4mXtp9tHtwA/RoFDp28HWJGubURpdXwTTEdr8AG0mRmlbldfn63VAFZOHWD1WCNdzvisHrQLL+6O4EX9Dltjh3N+tx+tWR1sR8DNLDojVtf/qFwYM2BZgPZvTKTTEd5AlBpiUrk3HThU0B9gstAvQQgF2UHGrTbM+x9iRojrkhAHk3F3Ddty+ePpDCN2Zdw7b6QJ1DtDCIax49Iwz6cg4QDic9IThEJxwxuB4h5AqjxJy0QxQvUlUw3KTH5wf2naUDg6zXa7JW3puC8dBtsDI67+7Y9CPxVJ80PF7OX6NyjPdgU4g1Heq0Tr8XEz+Urg7OYl5utIntsL3pLaAazZ/z87lhpn7uj+M8CzhQCe2MzCJxe/nmhZ5u2yje1VDPq7TK7NfT7j8P5GwfsAkNe43tJp6oOJL+0gzcZ5vqp5FZ/uRDf1Sac92SLPCWjgI1odzzWG2vqzaLApOrgEhJKl+b2KHiJ1aT5Ak0jiDuOFYVL/kjHYC2XJuFz7PFyNS1kmQShXsGMQlTjy9wYBzZhwcvvQDrztmMh0OLv2RjrsvQiEAU29Ds+xYcRzlLItSqlTgWHWJvDiaQK/jP2LQpoCkCIhZRoOMnsoc0nr0EcV+op8GdtBtrAEW0lzcgXw5aAGZ3+m1y7++OAxOAV3421mUPN+YdvAocIE+U1sD23fFLvLt73OCa5cxDhuNgrOzjh+g6uepnHn8cyf0Sq+F1H2cM3l7nlHF2SHWc1CW52WCqoj1OGbA8CwV9f9QD389+kg0NtUYOPOHhFeKfAcV/Fb/iRCJSaIdbv0ZG1soFWjiKKMfEWR4+skP4xg/LMXx05p81EtP9M9NJXBWN7pdbzTau0x7kTW2IcOBrhk6Cz1nXP8eKRYLSL/uTlFE4/gIIRbH1CipitwX4vCm3mlk9Qafai/lMqmFlieK47RMoP01W0KWpjGz1F7b5aIS9dwPgiNjCNHP9c6hHRhvLrfnWJqK1GMKHjIh0q10d6ytomkvznd12VSR2XlsEIDkjzGoO2+kOPFYv/Z6pJMjWjxqztGfmdv7XV18amv7CR87WK1GqBwlVHmAppyeu7gbKcxsCfY879nr3S9vEJ6/O2sVV/eWpmT2dHlO6TgRnr3pxZz6bVv7iesxUL0WFvdJr/0myEJzzX6gWmH1l65R/MY4eYxYCPbunGjq80PA0W7sfbqcGXX2MVt/02xnPWBuEuEYmPjT2Zr9RIRjUShZ09dQG87jnOtOMzqckHpjRrHbcwP13uih3Y1acB8pY1V6I236CButhOEYjRll6PQOlOGeP6RixyWo5llk5JnHYucx2W/a/L2BP15oheeTo5BHxNlzmEz7mxswshDzTneFYHSlwHxCskOjoGsdXSs/+cjaPpujh2M07bYmLbUj4yrk5j4uYeEbl4b6rTrgTeVpT0ubYyW99vOwuUTgOJs72A80sDj8RKPhspaj+482KCQsOJp25Mycuia3hCNHMBt0k6JeXoCr/w/cELTpiyQAAA=="
  },
  "ace_context_chunks.py": {
    "sha256": "8ab7629c197c5f557ba20b1e978791db23fe5ef16044f76a5eac0e227b9610c2",
    "gzip_base64": "H4sIAAAAAAAC/60825LbRnbv8xWQ9gHADAhyKEtrcwaq8irazVYSW2U7qVQxLFQTaJKIQIAGwLlYUVX+IX+YL8m5NdAAQUouRw8jsi+n+5w+9z7Nly9ffv/u/eTnRh+cdXksUp1OkrJo9FPj6KeDrrK9LpraUUXq7LWqj5VOA6dW+0OuJyrPtoVOnbrJmmSXFdvw6uonfax17TQ77SQ7nXyE7rWq62laHffOY1l91FUdOh9yVUwPVbnWTqo36pg3TlM6RelkxUZXukh0ePWD1mntFMf94RlWTDL6D7e4yXIdOJvN/qC3gXPIj7ycfspgI8XWefev//C9Ux2LBvbubMrKmUz0k06ODQB9+fLlVbY/lFXjqGp7UFWtzfedqnd5tjZf/7MuC/N5r5rd1aYq984BPsEgRzo+YId8rnfHJsvbb8c14Jfour4yTYSKowCnQzvK4IPN9YbXIFzDGmircrNQUlaVzlWDiGdFGh+0+thB1rlOmipL4s0xz2PY+JbgATFyDUfy/S/vnch5NZ/NZlc//fjjL/AFN+7FMS4dx35Y6brMH7Tnh0ASPPDl7erq6grOBtBS89dvPETcX1w58C/NtrpuIqFXKAN86nvMmh0DpwlhedCF51Zr16cdNZVWe4aC//Bw1nmZfIRzd7JGV16u9utULXggbEul3u1s/s01/vGDtev6C14+PB5SIIdH03ntSjfHqpDthTv9xJ9gZ4zJYwVLxHiutLngQeVHLThhQ9Tt+67RSFlVPUf4PUS04vq42WRPnhs2+4PLS7bDQgaOcuPhCmEKh117tEQABwY0jeaByvPyMS5UEf1V5bX2b9z/KE4gVfqQq0TzRmTruXouj43XlI3KA0cENPpuFgBrVtusiN4AX6hUR+Hrb6H/qOvoh7IAZtGFrrbP9EUwpS3VkSdAAgYQ4GzeSbZxPFU8exnwT90okEXvIViXZe47cFxF2Ti9Hi8rmmCTl6rx2wEoL2FWA6cCUbwHnw76AQ+ZV/dbDpAJt7P7SDZ0H72ZzUz77B73dc97xEb+dH+Ln2XCZH5tWmd+x1uVymrt/Buu976qyspzUaGYSbDi//73/8BKtSHh2+i2ZiLiPr2ZEMZHclcaeme12xJoQAU+FyCE3yF0jWJ3H1EX4UQNl/b3c3msEu3sj3XjgGI0O3RqDZtOzep/cv4OGGw10KJMde6kx0o1WVnUsO0kP8L2Nejqg0pTVMjOD/oBhuqnBJFHNVnpX+EEGiSFOoQEcq+esv1xH9GxwVHCZoRMPm1aJCSNKlRXHhOGevw7ok4kEOAoYJghEvbc334Zb4OPORrYl5Nr9QBmJAf9CbtWBVCjdkqG6ahjmpVCjgS21ES3uJ6QWvYCShHWgS/ePCDEEp3lHh8V79OfIjRf4KiDSrImA9FYYvMNDECgGXLDchbQOpPbFYOlbSBTU3eliq32aIS/6rDHjUW3Cxuy7I0HgUWCptnqzjmUeZbQANNDnwkMfARWc1CCZW1oMYsM5LQ5K6fNV8hp04llJ5mz++aeaDYlM4I4N4gz7oLg4IcXUQ1mSKderRuPei4K4l+QjUDVOWih65bheRuBcywyYFE09giTXA/AIwPORv5F6yZnPziAWz4ke2m9VwC02LIYTpDiSzjEdkADrKybyLTfMIO30667s1tm0D+tj3uv37ZY+R3Z8hK5z2vBCfMHvPpwcm/qLov2WdFNHawsIJjJJpl/LaA7AMANefl2ly1GFF/pbMB3y9bgZ6wN7dHXq4YagSVwKvo2z/ZZY/SORS7c6C4LENW8DLixR4UAsbF7QeBot4FM5dab+QCH8kCaLFryOTTc3Wc6RvQ+6g25j3bZyiaFQFrYHI0CR3uXzuCjfo7Y4XCahVrXXjMxyIhMPkcueHEPwHtpbCjntjBRHAcrALNm6JfUUQ7eqMeMSQS5uQ1wq9MpWCkSnOU4GVejW+4A93ftsYEH2UWs+QuriZl/E85m8+seXtM+vUmvC57omfD8uNCqinmGOwTrrlWOGiWNUXg7WiDjhuoA7l6KTOx3Ss208kL+1XA0cbY0747Fx1YLkngHngrWPh6+Bv9ZV+jz/ZYdSFACkhYSwe4YQOFVTQCAiQNngZqgqifGYzO9RtXe0cDYzwhImU5oNpOpHQHhi5YggI2ImeK3TktHBsLAoPbJRdfvyV1kgYsmIybw7kLJVxjjLtYBjDocG9PJCEgbjYA/Qe/MrH8ueQGx2ZK7GNnm555//Mnl8C1GWroLUlAukSbeVAoUsrtgQrmCXAy6KDZOyML4jVf9XZDG6EaJV+miP9W14rfANYIUGw5xF+YTrEkEhGXo/8/i/2IgEJPZ7wUiqlFBXUX1hiMF8upV/qie63ieRr9UYEXS5vmgI5cM36t558HV1YuILBowGcLBMOagIex5Ec2N7SsOnXHEMX4IDrz3ZR+TpwAnQihbQtz18R9/6/ksJlABkIIf2mkIkCFKiMuNB/5/Iat0HRG2hiCSnts1ukHXeoAgDgQ2+PTZPx3mYhDu+i36XR97CuDx8pDAxUgdho7YkZ+PhwPbZDodC0i9oCg/cGSyjWY3ysRhlAWIQby9rs/ga28tinhPC4EkAe3YOF63OxkJjLE5pjyD5RMw8fs9fwhZRgtj6Zqwss5PVvPcbr0Q40ND5/BRPZjPFL27fh+3lh9aJFkPey5TIz7AVgzINg+A+xPQvbZ2s8AsGOl7NcUcgQPRJaibrwstKWsg8eV1YJ+DsBBgLVkcllk7/OTDj8Z5wBztYD76v8il5AOLOyhTXrRHP3KCO8Aw7YAhmLo9VbQuNNUpi/xZWFbIwdTgfAA3WSmSgJp5iNVM87k13H9MM2ADTqWwHqLsVFx+lMifR1cZEBj0LqZSIsmjyHo0YIuWl3R4TOYgYvhTj62De8ODQ85MhGDCdeWJkHM6KkzKwzOelznpIchOIfLyJ/0vot42Lym/H2WgQzNBbtD7SZ3HHSa4mOMwP4c+J7K0ieAg1isMZq5ghLwr4e47VZRFlgDYlhnw7BAC+ujTffaEUPEUQ+dvvH0IJjCTRiuhWqwedCrgDDZw1rQnTtIhBLCcyU5yl+i9IagJWHJwADHJSeI2QXFzSI5rDp67RF9YHQtv6XJiEiRvUpR1A1E4fiRJRCpRO/7J3ACY2Bw3NKiKW9hRHrX3MCiBuXOcnywU/HdI9nF9+ybXPJdo6a8CSr0S4zGNWQwtK8oDu2ATRauNXI2Md2e9141CcxVRbgtsaVp7JAbtWJ9MMCfA/F5UYuayUWJ8Y+Y2F7irz/Z9d/pC6GigOmmpaxPMJjs6POuIKaOCfGtFMYRwf1dp+VistYJgJ1iu/JvBlkGBYrxAfX0oJ2Ev+yXa+MweeVQWVpSujOgMluyf0ogJhwbf+gv0U3NdeDTE594b07saWjFyazzwU+pfK/p/D8fKOdHr67kv+0FRiSSFaMEeyQAGiEXAe+/mmlxrDWy1V/GDrmp0mOfAcqo51hAjoSzq1A1aVcGn2lcdgZw9mQ9b0E+ZvcclhkWYZwP2c8SChB9+ev/z+18Cywh1H0/hWtrN3sxQ6YUFUD04HTyK1ckiwIyA3w9ymYEmCPhJV9OWyULnnZXqkrRdLfa2FgvHObFeLu/vEExgXIFeG/vHd2Sws+KYNc98R3OsSVX+elQ5th0LOK1sk+k0dL/GvNq5k74NJ3EfNA7NH0VtVfmbLjpLVZMVjtuZ7s0AyKj1su4Z5PSHS98NDdxgQMAb6akiASU9LyJe4WtcB2PLIKZCU4GLWhoF+X95iukq+uTi1iDwoRWZrVzRfXKR8Zndou5uwlhCkjx27MgBtP1JbBBP7oRLvU2Zp7oKLC+UbjekfdpFDOPSwMstXbsJIuzBUePlYAK6L7oETjDtA5TGVS+ZfuBMZEyXXNmDjptS9tutS1kT+zrqRWR2cckr+Vu7LWMPzGECOF2DCIrNB/Pdj1xwke4WrHe2I0SGA9xb1Djhhi72goEmpWtCG9v+jjj/v8vNxXxeBlTrBWM9FsAdLF052cHB/rEDYcjmiC/edgy3jRCmqFHN+Zw9DPQ/YqSLnQwY3OB1u7wj82c5LoTaibcCeFuHR4YN3ZPWto1QHm+xxa9l/sroEqbE7EqjeyEFXyNyRDAdkS+b9GIZuzvZwTFYoOwTEA/qRTQqbZeOovOwv1ZAhFhLk3RYvRiP6FoG9sVQ037Jkw452TTcdi8RdXHXH2hpAsi7NvkuMBwbYKqaYwAr0jhVlRYpLSV7IuzjwyzGtDvFzRFeJfP7HJO5tpVGQC1tlgCDUmIF8Kn0nj/F0SClYI21+dlItQVlxBtHuJGMbNe4ax17nmwSRxhcYiKpp+qRIBcV3J1KGvA7aJwNb2CZOx6yIS9sQ41i0AHDM+6+tatLAGED6VQPzrH3AKyCxSG1lR9Dsd3bDSf7fBENl8XDdr8YpbyjjFE91HD7rKYgRQigc7zB67Acd8NOgP9k4JnMVAIKqqQbtAqLb9B5BF8QXL+qG2lJAd7v9yjT6ToRENB2Q/KJnLdiauTevsjvR02DUMBM/QOuwugynDQH4ESLZS+Vvurn8U/mthdfXfocAHk9SJiAX03GgPtfgC44bFVGYMM/nw4Z5O47LAYdPTIP1CeI8hD783vuwRGzwQD6HAHqwRzbxXQ3MzplSAq8FZoaMZuavbNFH3C+yefCOibBfyxYR9ZeVZasSGPLutuaUyyq5QW0g7+YYuwM0p2tdOMzylaWuuxjX/XCpeiiq3hlJac3SdHkXdWUUeNu2C0XYiTvmjIqdcNlVNhoFVEhGCzbSD56+Cfghn/+8d0/xe///b+sbz/8xQpZqnILMizJXLO0aeW442Rwz4uyAfTMDXkHvV5Kftaez6nrT597BWAmvS4ehVwArfo6Fi89KcFlmBqv1Vbg25HdNfvf8Oz4Ew97ScNerhazV+lnt3/rSfMkUdvmZruEmWWLYGnanWC0OIn3E5XsIBBq4ysZuISJq7EIeWDnePqpk0f+3WLUXcMJ5He07hkh7J6u0XM/eCXb/TgZf4DwtkEbA4bEMakDgY43z8EmP9Y7JtSd5B50D8qfnJ90UmLRkaKLsBr3qhqsbwObvmmwo6D8RlUdD5iJNwTjIDjs33+LM512GUqbdwzcrJBqEaFbuM3LNQgMd8fXPTXWZTjIk5JBtvs0dlSiIDNmaeBlTLdY8iD9tiD07Wtk29fFaH63d1qyL/u47jpySO/dGhb8OGTZdlgbZp5m2NiKsKE5PDe7suCMLqpfkGE4x43KcvAqNljrlMAyVFwLZ4frPhvSn2a/jvs1aN/ZSQdfA8gBgaiaw/nEM1hI/VZXLLj5Jro9FbiWCpeBnW6Bc4bmGNjUyMVICzQgW3nenxhPzXdlBL/LfQikMpkviUYhX197n05c00Xb8LmX0etdlYm29b+kFlrMz2mGnkpjPWzmhKcJArCtdqmtbQkC860fQUmj+ALp2rNLc3up7vnsujiAxG1vZx7mz7mVhwe3enJLSW/xKfa1pLpHAA1z5vT9iFeinD3kugFopeFvvvE7uAesdu9B3pdFyUl9BqaAh6Nbf7lYfLsKVY2gPAuUccMxF49T/fs335DDDRvm74DJN4u2XmP3m7ug61YXC/LLNEuy5tldzMLP7eqTCP/y6nLlk0Rt0ThBDeRPiuUQeNMcgDO6K1P4tgEPetnuBovaBMQUKSBFi95t0A2BZiXFTaZN2CwvH4Ndtt1JgdY3s9lsOn894wocq+3Va5/gqQQWlBwVFrWTJ2Yq3KF3CQAXCBCtPbbdgOf3aOdLEAr1+BwoLmer+2j21eRbY5KZmQLmEqAV4gwK/C5X2wgNIu79wNeO5AvgIKq/hAmrtxGCuA6/e21qSctik6XkDLbMluTZgbBR29WU9hjMglt/UJmDWyXqwDBW5AbS2yh8/ZolegSXbpypmKHCm0cNdANlatdFKtT7eKw5eMUHLDSfBfDtkE3nXD95yvr2HhGVEs4FoQCDB7iZFQIAgzJoNalTrfbxIxxp+QjMsmmCCjcV5KRngor/SzQ6BMFjlrYuv6hh6phQx3Q6v8PCLuq5oSZGKUK4S2qe5GPKFkuoxnuY09cRbUpAVGdBVOdBIDdwERGEkLS3YE4MuT5pHUkl/hXrzsFrgEVq4MjnEpyLdQkOnlzSmTTuowoe11HvcJlmd+scSaOuH9XN+vpxzRkGFWhADfWgH+B/awx7NlzoTerP0+paqxu9vtZrfzo3/FtVHeeyRhtRY9cwg+av2edHIaEV/begv/7cMatU2IB5jEC30z5wsz64XBtfzJc1ZRbKVf0HMmvOr8dMU2mKaLPUaUD11BkXcYIwFirPftNCLwxD85wTgeiirY9Znh6l0h1Yt4k8QhjLI7k6knbm33j43bud4Dq+1K9Xp/Z0NghhpdowIJSCTy4zrFUvR9/Z2LvIpzEQIE7XG+gDamhl3fW7xIWDAWufqgXZzmKFH34JXFqvCxl4DmapYZAF8QCxaEbpsVw/gKvHqV0aSsCnqOng1MR24lJ14y749PGzbzQKv/KKVVWp55pi5IBMXi3yKsWb/SjujrhTbAF39UoCV1YFsFhEnuq/iNqr6npMYt6DD/MswaOpIKesVNG4djEOKrLfdAU6yxvLMIM4DhTeq3lXXc7KSW5JC4cKT2l7fdSN/PN1pkj7V/qBoiWG9YZyvzooOLyUheFd0urTAu0BFSEOMi+kLoO18W6totRVYLfRfrtsEVFyqRbrldQuqFFcFuvJeQWJdqBf22uX7lsnb+Hb2QvesRTDm8+ru6LcY5JR1HcPn64iWCvQ2+bxCiip2/C15BkHZaur6St/UJJuVXR7S1lt1d1/nNayrmiL0cXCcfsfKRWmgoCf8IYD+XrDX9v68fnMcugx2m4UJnzRDwId7exUvuHnl1JjYB5PHZv2DQW9coIZWiU7R6dbK+huqmzfE9chiYg+xlIMKLRMWGqoVr9tJl3aWvC35qx6LHKD647SB3V4csNWrgXi37PJ70kXglhdnWRXzDbGH0YgV3L+jcoQ4AiLhkjipFmdqCq1BIc8RWQgz/ZpWBMwY5rPq8DydBJxa8Zo49vl/E+Lp+Wr1ZJV8KonsGJeMJdQk7tpccDfwfRiiVogDoScPliMROpE3BrO9UhPPCjxkuGDipLGFfqR2CEcCnufGRdKJJ8RHfRO7MMwukCNtZ7olPViwOZEq26x1XI9GWOY0WmjI8fVGNHTOlnMcvQwNK8HRpCQaq1X/pdwpHTXl1EZb5aqr1c9lr5UGNw3E/khqg7R2eiHm7BUyV24H9Cy26VE4vUoKiGYbKsS+Mu8/3Y/9x/A5IcAFuKgmOnoBxIiI/qWoqBq2Nv5bHZNDihE8XOvOixxi6tpLh/I9TJfSPrNkIFDKTFCU5vyNMr2RpnRm0ZlRfKdXTD2PpDC7AJEipvQmeKG9Zm3FuS7EWIRokyeGn+tDoywca4YUfp7BtZeFXzr95Dpx8gjNJZnXLqVc+9MXlFou67N0PNu3cp/++bcY5HeP1RG9LZ/WF2Ny1AP+PEYuJ9BgSvNBIdYik7SC4ksKYz7mR+2byDErZ20pLU1ZnCwbJdZbdpk+3WlQ+c91rFNDlgZ1hV6bo+Yemg0+Hs7VWuLb0NLWZO7YR7iENW6JIMV3ICUYy4DUJaKbjGueG0XsbMQQGQ/xWl8swEfmBNvQzsS4OnXOA+iAFrdXdB/gYvZaCwX1CoF/PZ8J7igoXiS9J0SHu4C/5597OMYgRUappnaYpFxlnAV+x3+hII6NiXIV5Y4Sa6yPb2a3ZUVnGuSHCuVPAcOkxcJV1Odc/JMd5P92sHQHXj+WN78R+7l+hdudMmIBXGjF29jN07e+C3VSK0Da1zwNe/MFYbtd37xwgkQPb3NGbl9Gr3dEb8DYIyXT12+pBqLdOSS1b7u6Spx2tKOXkoX+keTuUwYIxVWPTjdAvstucwIAsR9dkhlDoIZQ6euid/k/UObwx8p+5IpMpR/taO+a8v0nawxdw4K6LgDwedVXOtoA7m6OR+Snr7GkAvdDf9Sg6my3LifWkb8zMXIvAIZi/q45udjH979Szz/xh2H4e6zJ3q7w/PD19de+Odrm7hc4nQjddcXQCN71lQeWDjLdmNBu4LNpX/w4UGfAIy7374ryLP1/vAK/FKNU9bUdvvd/OP4XBjqjrw6IGzQv8Y4j05sKYrRQoPzIxJyYO9ymEJZDUKy0zMkd/wTTe6Jk3Wg4s9JOvEVgxxW23PvjfSuLp1Ue/gszIxeDcrI5rYuhdAJkskj9JVPYCRv+JJVXrF6IwH7hWsl6zG2LrbNjizZt2EfTC8nYQMewhgpTu5jjZwwos2kerlnzT7xVHngGdi1JrGhAdhNqzZlYA3RYJLHgT8EIJ5cl2w7RWbszq6DYSYymQK7MImRcxe2BYJTPF9m99n/w7KJr+foVRA9Q7ZeCZ0Swp003Mdb/92ie94Ixax/Y9au54Rb7rvN0xB5CdJahPEXGW3QEugnpiKl4SJLBkd8n7ZcMRpR1xYDmacCZ3W7WLERfmZvkfmVMeuleblJ3CH8cQiTlDtE5mekwu+rLeH4Ab9VXgq2usrolwaiOE7LJI79O+CLCLzNNI2RQ2hgjSObCHh2D1FBistbnrT1gAdm0Eye5rlc0kfVzdSuZH3PnUzY4rgBKS681h9AHZ9VYtXnuSnda6DhNPM7PEaYGAS52IH8ulf03ezMmpxeujj3zYW1MXa7ODl8/e2ZlfGpk4XuhUXsh9TgSmcJprr6z6VX7YLy0vo8tPaufsJX+cMtoNFAliUfwKWnhrAMenyBiFfPCxjyBU4FRjtZltjl8uka9xUgvDCiDLr4FIOy7DGKQR1/W2xsaalxAEtMD+kQNOoXoKl2T1bmzCngPLJwWxDJi+Mv2rTn8WYWfDcLbuez1ZDXT+GwaRhjF7ndgcE1CCqRFGfW3eMI/BaKsEb8PMC1X03VCMY8r6bBUmBCn0HEAoHAv60gvMuN/Vwrt9nXJTIV3+hZ6pUarVdvw5db1D98PmVVIA8QYuJbJXoXIjya20VtXWVdV5FH8OJPNNKcH8c6d19ThMdJna6a6GwVEQ08KSKiVruG6PfUaVoH+qXXDW1JERMb64ouF40GFzGQGiRmH7pibOuLeoRsy4eoVb70Tej/V/nQGK+gUurfgFnbWAjprNLejjgjjEPJxDjgWUF8nstooAC3UhODUVzJaP0WnwA2P8ZH5Tyw5Zj8gzgGdOIYTXsc48+XoIm/+j8MnFZZHlMAAA=="
  }
}
# END EMBEDDED CONTEXT HELPERS
TOOLS = Path(tempfile.mkdtemp(prefix="drum_chunks_"))
for name, payload in EMBEDDED_HELPERS.items():
    data = gzip.decompress(base64.b64decode(payload["gzip_base64"], validate=True))
    assert hashlib.sha256(data).hexdigest() == payload["sha256"], f"Helper checksum mismatch: {name}"
    (TOOLS / name).write_bytes(data)
modules = {}
for name in ("electric_full_song", "drum_probe", "ace_context_chunks"):
    spec = importlib.util.spec_from_file_location(name, TOOLS / (name + ".py"))
    module = importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
    sys.modules[name] = module; modules[name] = module
context = modules["ace_context_chunks"]
ROOT = modules["electric_full_song"].discover_root(Path.cwd(), globals().get("ROOT"))
if ROOT is None and Path('/content').is_dir():
    ROOT = modules["electric_full_song"].discover_root(Path('/content'))
if ROOT is None:
    raise RuntimeError("Source bridge belum ada. Pakai VM aktif atau sel setup notebook 12 sampai bobot siap")
if not DRY_RUN:
    modules["electric_full_song"].model_python(ROOT)
SOURCE_PATH = globals().get("SOURCE_PATH")
if SOURCE_PATH is None or not Path(SOURCE_PATH).is_file():
    uploaded = files.upload()
    audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
    assert len(audios) == 1, "Upload satu piano asli seluruh lagu"
    SOURCE_PATH = TOOLS / ("piano" + Path(audios[0]).suffix.lower())
    SOURCE_PATH.write_bytes(uploaded[audios[0]])
DRUM_CHUNK_OUT = globals().get("DRUM_CHUNK_OUT")
if DRUM_CHUNK_OUT is None:
    DRUM_CHUNK_OUT = ROOT / "runs" / ("drum_chunks90_" + uuid.uuid4().hex[:10])
    drum_plan = context.prepare(SOURCE_PATH, DRUM_CHUNK_OUT, context=90, margin=6, fade=.58, instrument="drums")
else:
    DRUM_CHUNK_OUT = Path(DRUM_CHUNK_OUT)
    _, drum_plan, _ = context.load_plan(DRUM_CHUNK_OUT / "plan.json")
    assert context.instrument_of(drum_plan) == "drums" and drum_plan["context_cap_seconds"] == 90
    assert drum_plan["original_sha256"] == context.sha256(SOURCE_PATH)
DRUM_CHUNK_PLAN = DRUM_CHUNK_OUT / "plan.json"
print("Runtime dipakai kembali:", ROOT)
print("Plan drum:", DRUM_CHUNK_PLAN)
for c in drum_plan["chunks"]:
    print(c["index"] + 1, "model", c["model_duration"], "detik; core", round(c["core_start"]/32000, 3), "sampai", round(c["core_end"]/32000, 3))


## Generasi tiga konteks dan penyambungan

Jalankan sel ini lagi jika satu chunk gagal dan VM masih hidup: chunk selesai dipakai kembali setelah verifikasi. Tiap chunk memuat model; margin/reload menambah waktu. Tidak ada fallback otomatis ke durasi/seed lain.


In [ ]:
from IPython.display import Audio, display
if DRY_RUN:
    print("DRY RUN: plan saja; tidak menjalankan model atau stitching")
else:
    progress = context.run_chunks(ROOT, DRUM_CHUNK_PLAN)
    finished = DRUM_CHUNK_OUT / "stitched/seams.json"
    if finished.is_file():
        drum_seams = json.loads(finished.read_text())
        assert context.sha256(DRUM_CHUNK_OUT / "stitched/drums_full.wav") == drum_seams["stem_sha256"]
    else:
        drum_seams = context.stitch_run(DRUM_CHUNK_PLAN)
    print("Sambungan:", [(s["index"], round(s["center_seconds"], 3)) for s in drum_seams["seams"]])
    print("Seluruh sambungan perlu didengar; flag numerik bukan penilaian groove.")
    for name in ("drums_full.mp3", "mix_full.mp3"):
        display(Audio(filename=str(DRUM_CHUNK_OUT / "stitched" / name), normalize=False))


## Simpan hasil dan kirim feedback

Paket feedback berisi mix/drum penuh MP3, cuplikan tiap chunk sebelum stitching, cuplikan sambungan, serta report aktual tiap generasi. Paket result juga menyimpan WAV drum tiap chunk dan WAV hasil akhir. Keduanya tidak menyertakan bobot model atau piano asli.


In [ ]:
stitched = DRUM_CHUNK_OUT / "stitched"
if (stitched / "seams.json").is_file():
    for filename, full_result in [("drum_chunked_feedback.zip", False), ("drum_chunked_result.zip", True)]:
        bundle = DRUM_CHUNK_OUT / filename
        with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            archive.write(DRUM_CHUNK_PLAN, "plan.json")
            archive.write(DRUM_CHUNK_OUT / "progress.json", "progress.json")
            for path in sorted(stitched.rglob("*")):
                if path.is_file() and (path.suffix in {".json", ".mp3"} or path.name.startswith("seam_") or full_result and path.suffix == ".wav"):
                    archive.write(path, "stitched/" + str(path.relative_to(stitched)))
            if full_result:
                saved = json.loads((DRUM_CHUNK_OUT / "progress.json").read_text())
                for c in drum_plan["chunks"]:
                    folder = (DRUM_CHUNK_OUT / saved[str(c["index"])]).resolve()
                    assert folder.is_relative_to(DRUM_CHUNK_OUT / f'chunk_{c["index"]:03d}')
                    stem = context.verify_chunk(folder, drum_plan, c)
                    archive.write(stem, f'chunks/chunk_{c["index"]:03d}_drums.wav')
        print(filename, round(bundle.stat().st_size/1024**2, 2), "MiB")
        files.download(str(bundle))
else:
    print("Belum ada hasil lengkap. Tidak ada audio hasil yang dibuat oleh dry-run.")
